# TCN MODEL FOR MARS PRESSURE PREDICTION

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    Dropout,
    Add,
    LayerNormalization,
    Dense,
    GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    CSVLogger,
    ModelCheckpoint,
    ReduceLROnPlateau
)
import gc

# 1. EXPERIMENT CONFIGURATION

In [ ]:
EXPERIMENT_NAME = "TCN_Exp01"

TRAIN_PATH = "train.parquet"
TEST_PATH = "test.parquet"
SAMPLE_SUBMISSION_PATH = "sample_submission.csv"

TARGET_COL = "PRESSURE"

# Your data is recorded every millisecond.
# SAMPLING_RATE = 1000 means use 1 row every 1000 rows = 1 second.
SAMPLING_RATE = 1000

# 60 timesteps with sampling_rate=1000 means about 60 seconds of history.
TIME_STEPS = 60

BATCH_SIZE = 1024
EPOCHS = 20
LEARNING_RATE = 0.001

MA_WINDOW = 1000
MISSING_DROP_THRESHOLD = 0.50

TCN_FILTERS = 64
KERNEL_SIZE = 3
DROPOUT_RATE = 0.10

VALIDATION_RATIO = 0.20

print(f"--- Starting Experiment: {EXPERIMENT_NAME} ---")

# 2. CYCLICAL TIME FEATURE FUNCTION

In [ ]:
def extract_cyclic_time(df, time_col="LMST"):
    """
    Extracts time from LMST and creates cyclic time features.

    Time_Sin and Time_Cos capture the main daily cycle.

    Time2_Sin and Time2_Cos capture the second harmonic,
    which may help with the double peak/trough pressure pattern.
    """

    if time_col not in df.columns:
        print(f"{time_col} not found. Skipping time extraction.")
        return df

    print(f"Extracting cyclic time features from {time_col}...")

    time_strings = df[time_col].astype(str).str.extract(r"(\d{2}:\d{2}:\d{2})")[0]

    parsed_times = pd.to_datetime(
        time_strings,
        format="%H:%M:%S",
        errors="coerce"
    )

    hour_of_sol = (
        parsed_times.dt.hour
        + parsed_times.dt.minute / 60.0
        + parsed_times.dt.second / 3600.0
    )

    df["Time_Sin"] = np.sin(2 * np.pi * hour_of_sol / 24.0)
    df["Time_Cos"] = np.cos(2 * np.pi * hour_of_sol / 24.0)

    df["Time2_Sin"] = np.sin(4 * np.pi * hour_of_sol / 24.0)
    df["Time2_Cos"] = np.cos(4 * np.pi * hour_of_sol / 24.0)

    return df

# 3. TCN BLOCK

In [ ]:
def tcn_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    """
    One residual TCN block.

    Conv1D:
    - learns temporal patterns.

    padding='causal':
    - prevents the model from looking into the future.

    dilation_rate:
    - allows the model to see wider time history.

    residual connection:
    - helps the model train more stably.
    """

    shortcut = x

    x = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(x)

    x = Dropout(dropout_rate)(x)

    x = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(x)

    if shortcut.shape[-1] != filters:
        shortcut = Conv1D(
            filters=filters,
            kernel_size=1,
            padding="same"
        )(shortcut)

    x = Add()([shortcut, x])
    x = LayerNormalization()(x)

    return x

# 4. BUILD TCN MODEL

In [ ]:
def build_tcn_model(time_steps, num_features):
    inputs = Input(shape=(time_steps, num_features))

    x = inputs

    for dilation in [1, 2, 4, 8, 16, 32]:
        x = tcn_block(
            x=x,
            filters=TCN_FILTERS,
            kernel_size=KERNEL_SIZE,
            dilation_rate=dilation,
            dropout_rate=DROPOUT_RATE
        )

    x = GlobalAveragePooling1D()(x)

    x = Dense(64, activation="relu")(x)
    x = Dropout(DROPOUT_RATE)(x)

    x = Dense(32, activation="relu")(x)

    outputs = Dense(1)(x)

    model = Model(inputs=inputs, outputs=outputs)

    return model

# 5. DATA LOADING

In [ ]:
print("Loading datasets...")

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# 6. FEATURE ENGINEERING

In [ ]:
print("Applying cyclic time features...")

train_df = extract_cyclic_time(train_df, "LMST")
test_df = extract_cyclic_time(test_df, "LMST")


print("Dropping columns with too much missing data...")

missing_percentages = train_df.isnull().mean()

bad_sensor_columns = missing_percentages[
    missing_percentages > MISSING_DROP_THRESHOLD
].index.tolist()

columns_to_drop = bad_sensor_columns + [
    "LMST",
    "LTST",
    "SCLK"
]

if TARGET_COL in columns_to_drop:
    columns_to_drop.remove(TARGET_COL)

print("Columns to drop:")
print(columns_to_drop)

train_df = train_df.drop(columns=columns_to_drop, errors="ignore")
test_df = test_df.drop(columns=columns_to_drop, errors="ignore")


print("Converting object columns to numeric...")

for col in train_df.columns:
    if col == TARGET_COL:
        continue

    if train_df[col].dtype == "object":
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")


print("Interpolating missing values...")

train_df = train_df.interpolate(method="linear").ffill().bfill()
test_df = test_df.interpolate(method="linear").ffill().bfill()

# 7. MOVING AVERAGE SMOOTHING

In [ ]:
print(f"Applying {MA_WINDOW}-row moving average...")

time_features = [
    "Time_Sin",
    "Time_Cos",
    "Time2_Sin",
    "Time2_Cos"
]

do_not_smooth_keywords = [
    "Time_Sin",
    "Time_Cos",
    "Time2_Sin",
    "Time2_Cos",
    "TRANSDUCER",
    "FOV",
    "OFF",
    "STILL",
    "TILT"
]

candidate_features = [
    col for col in train_df.columns
    if col != TARGET_COL
]

smooth_features = []

for col in candidate_features:
    should_not_smooth = any(keyword in col for keyword in do_not_smooth_keywords)

    if not should_not_smooth:
        smooth_features.append(col)

print("Number of smoothed features:", len(smooth_features))

train_df[smooth_features] = train_df[smooth_features].rolling(
    window=MA_WINDOW,
    min_periods=1
).mean()

test_df[smooth_features] = test_df[smooth_features].rolling(
    window=MA_WINDOW,
    min_periods=1
).mean()


final_features = [
    col for col in train_df.columns
    if col != TARGET_COL
]

print("Final feature count:", len(final_features))
print("Final features:")
print(final_features)

# 8. PREPARE ARRAYS

In [ ]:
print("Preparing arrays...")

X_train_raw = train_df[final_features].values.astype(np.float32)
y_train_raw = train_df[TARGET_COL].values.astype(np.float32)
X_test_raw = test_df[final_features].values.astype(np.float32)

del train_df
del test_df
gc.collect()

# 9. SCALING

In [ ]:
print("Scaling features...")

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

del X_train_raw
del X_test_raw
gc.collect()

# 10. TRAIN / VALIDATION SPLIT

In [ ]:
print("Creating train/validation split...")

split_idx = int(len(X_train_scaled) * (1.0 - VALIDATION_RATIO))

history_offset = (TIME_STEPS - 1) * SAMPLING_RATE

print("History offset in rows:", history_offset)
print("This means each sample looks back about", history_offset / 1000, "seconds.")

# 11. CREATE TENSORFLOW DATASETS

In [ ]:
print("Creating TensorFlow datasets...")

X_train_part = X_train_scaled[:split_idx]
y_train_part = y_train_raw[history_offset:split_idx]

train_dataset = tf.keras.utils.timeseries_dataset_from_array(
    data=X_train_part,
    targets=y_train_part,
    sequence_length=TIME_STEPS,
    sequence_stride=1,
    sampling_rate=SAMPLING_RATE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

X_val_part = X_train_scaled[split_idx - history_offset:]
y_val_part = y_train_raw[split_idx:]

val_dataset = tf.keras.utils.timeseries_dataset_from_array(
    data=X_val_part,
    targets=y_val_part,
    sequence_length=TIME_STEPS,
    sequence_stride=1,
    sampling_rate=SAMPLING_RATE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

# 12. BUILD AND TRAIN MODEL

In [ ]:
print("Building TCN model...")

model = build_tcn_model(
    time_steps=TIME_STEPS,
    num_features=len(final_features)
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

model.compile(
    optimizer=optimizer,
    loss="mse",
    metrics=["mae"]
)

model.summary()


csv_logger = CSVLogger(
    f"{EXPERIMENT_NAME}_log.csv",
    append=True
)

checkpoint = ModelCheckpoint(
    f"{EXPERIMENT_NAME}_best_model.keras",
    save_best_only=True,
    monitor="val_loss",
    mode="min",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    mode="min",
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    mode="min",
    verbose=1
)


print("Starting training...")

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=[
        early_stop,
        reduce_lr,
        csv_logger,
        checkpoint
    ]
)

# 13. TEST DATASET FOR SUBMISSION

In [ ]:
print("Preparing test data generator...")

last_train_context = X_train_scaled[-history_offset:]

X_test_padded = np.vstack([
    last_train_context,
    X_test_scaled
])

test_dataset = tf.keras.utils.timeseries_dataset_from_array(
    data=X_test_padded,
    targets=None,
    sequence_length=TIME_STEPS,
    sequence_stride=1,
    sampling_rate=SAMPLING_RATE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

# 14. PREDICTION

In [ ]:
print("Generating predictions...")

predictions = model.predict(test_dataset)
predictions = predictions.flatten()

print("Prediction count:", len(predictions))

# 15. SUBMISSION

In [ ]:
submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Submission rows:", len(submission))

if len(predictions) != len(submission):
    raise ValueError(
        f"Prediction length mismatch. "
        f"Predictions: {len(predictions)}, submission rows: {len(submission)}"
    )

submission[TARGET_COL] = predictions

submission_filename = f"submission_{EXPERIMENT_NAME}.csv"

submission.to_csv(
    submission_filename,
    index=False
)

print("Experiment complete.")
print("Saved submission:", submission_filename)
print("Saved log:", f"{EXPERIMENT_NAME}_log.csv")
print("Saved model:", f"{EXPERIMENT_NAME}_best_model.keras")